# 01 Model Training

Notebook для обучения GRU/LSTM. Лучший trial выбираем по настраиваемой validation-метрике, test используем только как финальную проверку.

In [1]:
from pathlib import Path
import random
import sys

import joblib
import numpy as np
import optuna
import pandas as pd
import torch
from optuna.pruners import MedianPruner
from optuna.samplers import TPESampler

PROJECT_ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

from src import config
from src.connector.data_fetcher import load_all_price_data
from src.models.training import train_direction_model

SEED = 42
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)

price_df, files = load_all_price_data(PROJECT_ROOT / "data")
print(f"loaded files={len(files)}, rows={len(price_df)}")

loaded files=45, rows=277105


## Optuna подбор параметров

F1 может быть высоким при перекосе в один класс. Для переобучения лучше начать с balanced accuracy.

In [2]:
SELECTION_METRIC = "balanced_accuracy"  # accuracy / balanced_accuracy / f1
EPOCHS = 40
N_TRIALS = 25
LABEL_THRESHOLD = config.DEFAULT_LABEL_THRESHOLD
HORIZON = config.DEFAULT_HORIZON_CANDLES

best_global_score = {
    "event_gru": -1.0,
    "event_lstm": -1.0,
    "full_gru": -1.0,
    "full_lstm": -1.0,
}


def get_paths(model_type: str, event_only: bool):
    if event_only:
        return (
            config.EVENT_MODEL_PATHS[model_type],
            config.EVENT_SCALER_PATHS[model_type],
            config.EVENT_CONFIG_PATHS[model_type],
        )
    return (
        config.FULL_MODEL_PATHS[model_type],
        config.FULL_SCALER_PATHS[model_type],
        config.FULL_CONFIG_PATHS[model_type],
    )


def make_study(name: str):
    return optuna.create_study(
        direction="maximize",
        study_name=name,
        sampler=TPESampler(seed=SEED),
        pruner=MedianPruner(n_startup_trials=5, n_warmup_steps=8),
    )


def make_objective(model_type: str, event_only: bool = True):

    def objective(trial):

        learning_rate = trial.suggest_float("learning_rate", 1e-4, 3e-3, log=True)
        hidden_size = trial.suggest_categorical("hidden_size", [32, 64, 128, 256])
        dropout = trial.suggest_float("dropout", 0.1, 0.5)
        num_layers = trial.suggest_int("num_layers", 1, 3)
        batch_size = trial.suggest_categorical("batch_size", [64, 128, 256])

        result = train_direction_model(
            price_df=price_df,
            model_type=model_type,
            event_only=event_only,
            label_threshold=LABEL_THRESHOLD,
            horizon=HORIZON,
            epochs=EPOCHS,
            batch_size=batch_size,
            learning_rate=learning_rate,
            hidden_size=hidden_size,
            dropout=dropout,
            num_layers=num_layers,
            selection_metric=SELECTION_METRIC,
            trial=trial,
        )

        best_valid_score = result["best_valid_score"]
        dataset_mode = "event" if event_only else "full"
        global_key = f"{dataset_mode}_{model_type}"

        if best_valid_score > best_global_score[global_key]:
            best_global_score[global_key] = best_valid_score
            model_path, scaler_path, config_path = get_paths(model_type, event_only)

            model_path.parent.mkdir(parents=True, exist_ok=True)
            scaler_path.parent.mkdir(parents=True, exist_ok=True)
            config_path.parent.mkdir(parents=True, exist_ok=True)

            torch.save(result["model"].state_dict(), model_path)
            joblib.dump(result["scaler"], scaler_path)
            joblib.dump(
                {
                    "model_type": model_type,
                    "hidden_size": hidden_size,
                    "dropout": dropout,
                    "num_layers": num_layers,
                    "selection_metric": SELECTION_METRIC,
                    "label_threshold": LABEL_THRESHOLD,
                    "horizon": HORIZON,
                },
                config_path,
            )

            print(
                f"\nNEW GLOBAL BEST {dataset_mode.upper()} + {model_type.upper()} | "
                f"VALID {SELECTION_METRIC}: {best_valid_score:.4f} | "
                f"TEST acc: {result['test_metrics']['accuracy']:.4f} | "
                f"TEST bal_acc: {result['test_metrics']['balanced_accuracy']:.4f} | "
                f"TEST f1: {result['test_metrics']['f1']:.4f}"
            )
            print("Saved model:", model_path)
            print("Saved scaler:", scaler_path)
            print("Saved config:", config_path)

        return best_valid_score

    return objective

## 1. Подбор параметров GRU

In [3]:
study_gru_event = make_study("gru_event")

study_gru_event.optimize(
    make_objective("gru", event_only=True),
    n_trials=N_TRIALS,
)

[I 2026-05-08 12:38:06,884] A new study created in memory with name: gru_event
[I 2026-05-08 12:39:13,726] Trial 0 finished with value: 0.5259157038624719 and parameters: {'learning_rate': 0.0003574712922600243, 'hidden_size': 32, 'dropout': 0.16239780813448107, 'num_layers': 1, 'batch_size': 64}. Best is trial 0 with value: 0.5259157038624719.



NEW GLOBAL BEST EVENT + GRU | VALID balanced_accuracy: 0.5259 | TEST acc: 0.5275 | TEST bal_acc: 0.5271 | TEST f1: 0.5057
Saved model: /workspace/data/models/gru_event_direction_best.pth
Saved scaler: /workspace/data/models/gru_event_direction_scaler.pkl
Saved config: /workspace/data/models/gru_event_direction_config.pkl


[I 2026-05-08 12:40:21,003] Trial 1 finished with value: 0.5294801591379539 and parameters: {'learning_rate': 0.00010725209743172001, 'hidden_size': 32, 'dropout': 0.17336180394137352, 'num_layers': 1, 'batch_size': 64}. Best is trial 1 with value: 0.5294801591379539.



NEW GLOBAL BEST EVENT + GRU | VALID balanced_accuracy: 0.5295 | TEST acc: 0.5165 | TEST bal_acc: 0.5157 | TEST f1: 0.4605
Saved model: /workspace/data/models/gru_event_direction_best.pth
Saved scaler: /workspace/data/models/gru_event_direction_scaler.pkl
Saved config: /workspace/data/models/gru_event_direction_config.pkl


[I 2026-05-08 12:41:29,163] Trial 2 finished with value: 0.5237143444367779 and parameters: {'learning_rate': 0.0008012737503998541, 'hidden_size': 256, 'dropout': 0.41407038455720546, 'num_layers': 1, 'batch_size': 128}. Best is trial 1 with value: 0.5294801591379539.
[I 2026-05-08 12:42:37,207] Trial 3 finished with value: 0.5275298383663403 and parameters: {'learning_rate': 0.0007896186801026692, 'hidden_size': 256, 'dropout': 0.4233589392465845, 'num_layers': 1, 'batch_size': 128}. Best is trial 1 with value: 0.5294801591379539.
[I 2026-05-08 12:43:42,020] Trial 4 finished with value: 0.5278990560359382 and parameters: {'learning_rate': 0.00015144860262751404, 'hidden_size': 128, 'dropout': 0.36500891374159283, 'num_layers': 1, 'batch_size': 128}. Best is trial 1 with value: 0.5294801591379539.
[I 2026-05-08 12:44:44,486] Trial 5 finished with value: 0.5286940117738597 and parameters: {'learning_rate': 0.0027051668818999287, 'hidden_size': 64, 'dropout': 0.4687496940092467, 'num_la

...

[I 2026-05-08 12:51:19,708] Trial 11 pruned. 
[I 2026-05-08 12:52:25,233] Trial 12 finished with value: 0.5264236534198512 and parameters: {'learning_rate': 0.00023948841281366131, 'hidden_size': 32, 'dropout': 0.11030095293953826, 'num_layers': 2, 'batch_size': 64}. Best is trial 10 with value: 0.5300115976922061.
[I 2026-05-08 12:53:29,318] Trial 13 finished with value: 0.5258885447098375 and parameters: {'learning_rate': 0.00010388784737788284, 'hidden_size': 32, 'dropout': 0.19752777921904, 'num_layers': 2, 'batch_size': 64}. Best is trial 10 with value: 0.5300115976922061.
[I 2026-05-08 12:54:32,527] Trial 14 finished with value: 0.5279937460545826 and parameters: {'learning_rate': 0.00020776823961235662, 'hidden_size': 32, 'dropout': 0.15612212620230284, 'num_layers': 2, 'batch_size': 64}. Best is trial 10 with value: 0.5300115976922061.
[I 2026-05-08 12:55:41,628] Trial 15 finished with value: 0.5313409281089889 and parameters: {'learning_rate': 0.0002886856153639143, 'hidden_si


NEW GLOBAL BEST EVENT + GRU | VALID balanced_accuracy: 0.5313 | TEST acc: 0.5280 | TEST bal_acc: 0.5278 | TEST f1: 0.5134
Saved model: /workspace/data/models/gru_event_direction_best.pth
Saved scaler: /workspace/data/models/gru_event_direction_scaler.pkl
Saved config: /workspace/data/models/gru_event_direction_config.pkl


[I 2026-05-08 12:56:47,293] Trial 16 finished with value: 0.5275724121731726 and parameters: {'learning_rate': 0.0003516455665329036, 'hidden_size': 32, 'dropout': 0.2587674845661954, 'num_layers': 3, 'batch_size': 64}. Best is trial 15 with value: 0.5313409281089889.
[I 2026-05-08 12:57:56,675] Trial 17 finished with value: 0.529668071112938 and parameters: {'learning_rate': 0.00024579089084489873, 'hidden_size': 128, 'dropout': 0.2819805326886606, 'num_layers': 3, 'batch_size': 256}. Best is trial 15 with value: 0.5313409281089889.
[I 2026-05-08 12:59:06,462] Trial 18 finished with value: 0.5303507200845604 and parameters: {'learning_rate': 0.00046854024680870806, 'hidden_size': 64, 'dropout': 0.34557220007572076, 'num_layers': 3, 'batch_size': 64}. Best is trial 15 with value: 0.5313409281089889.
[I 2026-05-08 13:00:18,714] Trial 19 finished with value: 0.527896119911329 and parameters: {'learning_rate': 0.0005341272299069856, 'hidden_size': 64, 'dropout': 0.3417329666387555, 'num_l


NEW GLOBAL BEST EVENT + GRU | VALID balanced_accuracy: 0.5322 | TEST acc: 0.5199 | TEST bal_acc: 0.5199 | TEST f1: 0.5197
Saved model: /workspace/data/models/gru_event_direction_best.pth
Saved scaler: /workspace/data/models/gru_event_direction_scaler.pkl
Saved config: /workspace/data/models/gru_event_direction_config.pkl


[I 2026-05-08 13:05:46,376] Trial 24 finished with value: 0.5265924805848761 and parameters: {'learning_rate': 0.0002691571862796827, 'hidden_size': 32, 'dropout': 0.274842680107939, 'num_layers': 3, 'batch_size': 64}. Best is trial 23 with value: 0.5322085529309863.


In [4]:
print("BEST TRIAL:")
print(study_gru_event.best_trial.params)

print()
print(f"BEST {SELECTION_METRIC}:")
print(study_gru_event.best_value)

BEST TRIAL:
{'learning_rate': 0.00017568957231151638, 'hidden_size': 32, 'dropout': 0.2591232378147487, 'num_layers': 2, 'batch_size': 64}

BEST balanced_accuracy:
0.5322085529309863


In [5]:
optuna.visualization.plot_optimization_history(study_gru_event)

In [6]:
optuna.visualization.plot_param_importances(study_gru_event)

## 2. Подбор параметров LSTM

In [7]:
study_lstm_event = make_study("lstm_event")

study_lstm_event.optimize(
    make_objective("lstm", event_only=True),
    n_trials=N_TRIALS,
)

[I 2026-05-08 13:05:48,323] A new study created in memory with name: lstm_event
[I 2026-05-08 13:06:54,459] Trial 0 finished with value: 0.5195002715915263 and parameters: {'learning_rate': 0.0003574712922600243, 'hidden_size': 32, 'dropout': 0.16239780813448107, 'num_layers': 1, 'batch_size': 64}. Best is trial 0 with value: 0.5195002715915263.



NEW GLOBAL BEST EVENT + LSTM | VALID balanced_accuracy: 0.5195 | TEST acc: 0.5155 | TEST bal_acc: 0.5146 | TEST f1: 0.4565
Saved model: /workspace/data/models/lstm_event_direction_best.pth
Saved scaler: /workspace/data/models/lstm_event_direction_scaler.pkl
Saved config: /workspace/data/models/lstm_event_direction_config.pkl


[I 2026-05-08 13:08:02,814] Trial 1 finished with value: 0.5288459562223821 and parameters: {'learning_rate': 0.00010725209743172001, 'hidden_size': 32, 'dropout': 0.17336180394137352, 'num_layers': 1, 'batch_size': 64}. Best is trial 1 with value: 0.5288459562223821.



NEW GLOBAL BEST EVENT + LSTM | VALID balanced_accuracy: 0.5288 | TEST acc: 0.5238 | TEST bal_acc: 0.5236 | TEST f1: 0.5115
Saved model: /workspace/data/models/lstm_event_direction_best.pth
Saved scaler: /workspace/data/models/lstm_event_direction_scaler.pkl
Saved config: /workspace/data/models/lstm_event_direction_config.pkl


[I 2026-05-08 13:09:11,286] Trial 2 finished with value: 0.5249621973956574 and parameters: {'learning_rate': 0.0008012737503998541, 'hidden_size': 256, 'dropout': 0.41407038455720546, 'num_layers': 1, 'batch_size': 128}. Best is trial 1 with value: 0.5288459562223821.
[I 2026-05-08 13:10:17,625] Trial 3 finished with value: 0.5276759105656443 and parameters: {'learning_rate': 0.0007896186801026692, 'hidden_size': 256, 'dropout': 0.4233589392465845, 'num_layers': 1, 'batch_size': 128}. Best is trial 1 with value: 0.5288459562223821.
[I 2026-05-08 13:11:25,799] Trial 4 finished with value: 0.5267547014695304 and parameters: {'learning_rate': 0.00015144860262751404, 'hidden_size': 128, 'dropout': 0.36500891374159283, 'num_layers': 1, 'batch_size': 128}. Best is trial 1 with value: 0.5288459562223821.
[I 2026-05-08 13:12:23,557] Trial 5 finished with value: 0.5269558260052556 and parameters: {'learning_rate': 0.0027051668818999287, 'hidden_size': 64, 'dropout': 0.4687496940092467, 'num_la


NEW GLOBAL BEST EVENT + LSTM | VALID balanced_accuracy: 0.5307 | TEST acc: 0.5222 | TEST bal_acc: 0.5218 | TEST f1: 0.4911
Saved model: /workspace/data/models/lstm_event_direction_best.pth
Saved scaler: /workspace/data/models/lstm_event_direction_scaler.pkl
Saved config: /workspace/data/models/lstm_event_direction_config.pkl


[I 2026-05-08 13:25:22,432] Trial 16 pruned. 
[I 2026-05-08 13:26:27,248] Trial 17 pruned. 
[I 2026-05-08 13:27:31,143] Trial 18 finished with value: 0.5305078027511487 and parameters: {'learning_rate': 0.00020207777615448064, 'hidden_size': 64, 'dropout': 0.11592872189609416, 'num_layers': 1, 'batch_size': 64}. Best is trial 15 with value: 0.5307067251934172.
[I 2026-05-08 13:28:32,632] Trial 19 finished with value: 0.5260089258188118 and parameters: {'learning_rate': 0.0005371817361778608, 'hidden_size': 64, 'dropout': 0.10180139976233493, 'num_layers': 2, 'batch_size': 256}. Best is trial 15 with value: 0.5307067251934172.
[I 2026-05-08 13:29:34,956] Trial 20 pruned. 
[I 2026-05-08 13:30:40,854] Trial 21 pruned. 
[I 2026-05-08 13:31:46,344] Trial 22 pruned. 
[I 2026-05-08 13:32:52,743] Trial 23 finished with value: 0.5256822819560462 and parameters: {'learning_rate': 0.00012988194329426509, 'hidden_size': 64, 'dropout': 0.2656693330641768, 'num_layers': 1, 'batch_size': 64}. Best is

In [8]:
print("BEST TRIAL:")
print(study_lstm_event.best_trial.params)

print()
print(f"BEST {SELECTION_METRIC}:")
print(study_lstm_event.best_value)

BEST TRIAL:
{'learning_rate': 0.00011210124205357184, 'hidden_size': 32, 'dropout': 0.2960678959228117, 'num_layers': 1, 'batch_size': 128}

BEST balanced_accuracy:
0.5307067251934172


In [9]:
optuna.visualization.plot_optimization_history(study_lstm_event)

In [10]:
optuna.visualization.plot_param_importances(study_lstm_event)

## 3. GRU Training без event-фильтра

In [11]:
study_gru_full = make_study("gru_full")

study_gru_full.optimize(
    make_objective("gru", event_only=False),
    n_trials=N_TRIALS,
)

[I 2026-05-08 13:34:00,213] A new study created in memory with name: gru_full
[I 2026-05-08 13:36:39,001] Trial 0 finished with value: 0.5244925548627639 and parameters: {'learning_rate': 0.0003574712922600243, 'hidden_size': 32, 'dropout': 0.16239780813448107, 'num_layers': 1, 'batch_size': 64}. Best is trial 0 with value: 0.5244925548627639.



NEW GLOBAL BEST FULL + GRU | VALID balanced_accuracy: 0.5245 | TEST acc: 0.5124 | TEST bal_acc: 0.5115 | TEST f1: 0.4055
Saved model: /workspace/data/models/gru_full_direction_best.pth
Saved scaler: /workspace/data/models/gru_full_direction_scaler.pkl
Saved config: /workspace/data/models/gru_full_direction_config.pkl


[I 2026-05-08 13:39:01,730] Trial 1 finished with value: 0.5233489642896387 and parameters: {'learning_rate': 0.00010725209743172001, 'hidden_size': 32, 'dropout': 0.17336180394137352, 'num_layers': 1, 'batch_size': 64}. Best is trial 0 with value: 0.5244925548627639.
[I 2026-05-08 13:43:23,672] Trial 2 finished with value: 0.5240768831339307 and parameters: {'learning_rate': 0.0008012737503998541, 'hidden_size': 256, 'dropout': 0.41407038455720546, 'num_layers': 1, 'batch_size': 128}. Best is trial 0 with value: 0.5244925548627639.
[I 2026-05-08 13:47:38,601] Trial 3 finished with value: 0.5223906814796129 and parameters: {'learning_rate': 0.0007896186801026692, 'hidden_size': 256, 'dropout': 0.4233589392465845, 'num_layers': 1, 'batch_size': 128}. Best is trial 0 with value: 0.5244925548627639.
[I 2026-05-08 13:50:33,732] Trial 4 finished with value: 0.5243905967848093 and parameters: {'learning_rate': 0.00015144860262751404, 'hidden_size': 128, 'dropout': 0.36500891374159283, 'num_l


NEW GLOBAL BEST FULL + GRU | VALID balanced_accuracy: 0.5245 | TEST acc: 0.5168 | TEST bal_acc: 0.5157 | TEST f1: 0.3725
Saved model: /workspace/data/models/gru_full_direction_best.pth
Saved scaler: /workspace/data/models/gru_full_direction_scaler.pkl
Saved config: /workspace/data/models/gru_full_direction_config.pkl


[I 2026-05-08 13:54:52,942] Trial 6 finished with value: 0.5227056178286844 and parameters: {'learning_rate': 0.00037507963596256056, 'hidden_size': 64, 'dropout': 0.31707843326329943, 'num_layers': 1, 'batch_size': 256}. Best is trial 5 with value: 0.5245314208108623.
[I 2026-05-08 13:59:52,764] Trial 7 finished with value: 0.5242715384924477 and parameters: {'learning_rate': 0.0013826083091896758, 'hidden_size': 128, 'dropout': 0.3916028672163949, 'num_layers': 3, 'batch_size': 128}. Best is trial 5 with value: 0.5245314208108623.
[I 2026-05-08 14:02:22,567] Trial 8 finished with value: 0.5216346872370936 and parameters: {'learning_rate': 0.0018832519048593583, 'hidden_size': 32, 'dropout': 0.23007332881069884, 'num_layers': 3, 'batch_size': 128}. Best is trial 5 with value: 0.5245314208108623.
[I 2026-05-08 14:11:20,509] Trial 9 finished with value: 0.5237438787473854 and parameters: {'learning_rate': 0.00015019490572374368, 'hidden_size': 256, 'dropout': 0.2975182385457563, 'num_la


NEW GLOBAL BEST FULL + GRU | VALID balanced_accuracy: 0.5262 | TEST acc: 0.5132 | TEST bal_acc: 0.5126 | TEST f1: 0.4332
Saved model: /workspace/data/models/gru_full_direction_best.pth
Saved scaler: /workspace/data/models/gru_full_direction_scaler.pkl
Saved config: /workspace/data/models/gru_full_direction_config.pkl


[I 2026-05-08 14:48:09,853] Trial 22 finished with value: 0.5248954306561385 and parameters: {'learning_rate': 0.00016491100297406352, 'hidden_size': 128, 'dropout': 0.4593587889326771, 'num_layers': 1, 'batch_size': 128}. Best is trial 21 with value: 0.5261701615056975.
[I 2026-05-08 14:50:29,884] Trial 23 finished with value: 0.523672691321943 and parameters: {'learning_rate': 0.00017211055149195508, 'hidden_size': 128, 'dropout': 0.45894087183361587, 'num_layers': 1, 'batch_size': 128}. Best is trial 21 with value: 0.5261701615056975.
[I 2026-05-08 14:53:05,672] Trial 24 finished with value: 0.524833790511744 and parameters: {'learning_rate': 0.0001131427189882484, 'hidden_size': 128, 'dropout': 0.46744847629170433, 'num_layers': 1, 'batch_size': 128}. Best is trial 21 with value: 0.5261701615056975.


In [12]:
print("BEST TRIAL:")
print(study_gru_full.best_trial.params)

print()
print(f"BEST {SELECTION_METRIC}:")
print(study_gru_full.best_value)

BEST TRIAL:
{'learning_rate': 0.0001649761306753644, 'hidden_size': 128, 'dropout': 0.3640192932347782, 'num_layers': 1, 'batch_size': 128}

BEST balanced_accuracy:
0.5261701615056975


In [13]:
optuna.visualization.plot_optimization_history(study_gru_full)

In [14]:
optuna.visualization.plot_param_importances(study_gru_full)